# ⚖️ Phase 7 — Legal Domain: Zero-Training Baseline Benchmark for Graph & Ontology Tasks

## 🎯 Objective
In this benchmark, we evaluate how well untuned `Qwen2.5-1.5B-Instruct` performs on **Corporate Litigation & Legal Knowledge Graph Tasks** with **ZERO training**.

We evaluate 3 standardized legal tasks:
1. **Task A (Legal Triplet Extraction)**: Extracting structured triples matching the `CorporateLitigationOntology` from case descriptions.
2. **Task B (Multi-Hop Precedent & Remedy Reasoning)**: Multi-hop reasoning across judicial precedent chains (`Court -> PrecedentCase -> LegalRemedy`).
3. **Task C (Legal Link Prediction)**: Identifying the valid legal relation connecting parties, claims, and remedies.

---
## 🛠️ Google Colab Setup Note
- **Target Runtime**: T4 GPU (free tier) or CPU
- **Dependencies**: `transformers`, `accelerate`, `bitsandbytes`, `torch`

### Step 1: Environment & GPU Detection

In [7]:
# Run this cell if executing on Google Colab:
!pip install -q transformers accelerate bitsandbytes
!git clone https://github.com/sagarsambhwani/Fine-Tuning-PoCs.git

fatal: destination path 'Fine-Tuning-PoCs' already exists and is not an empty directory.


In [8]:
import sys
import os
import json
import time
import torch
import transformers

print("=" * 60)
print("🚀 LEGAL GRAPH BENCHMARK — HARDWARE AUDIT")
print("=" * 60)
print(f"Python Version       : {sys.version.split()[0]}")
print(f"PyTorch Version      : {torch.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total           : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("GPU Device Name      : CPU Mode (Quantization disabled)")
print(f"Transformers Version : {transformers.__version__}")
print("=" * 60)

🚀 LEGAL GRAPH BENCHMARK — HARDWARE AUDIT
Python Version       : 3.13.15
PyTorch Version      : 2.11.0+cu130
CUDA Available       : True
GPU Device Name      : Tesla T4
VRAM Total           : 14.56 GB
Transformers Version : 5.18.0


### Step 2: Load Untuned Base Model in 4-bit (Colab T4 Optimized)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading untuned tokenizer & weights for: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"

if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float32,
        device_map="auto"
    )

model.eval()
print("✅ Base Model Loaded successfully. Ready for Legal Benchmark Evaluation.")

Loading untuned tokenizer & weights for: Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

### Step 3: Load the Legal Benchmark Datasets

In [ ]:
def load_legal_benchmark_file(filename):
    paths_to_try = [
        f"data/benchmark/{filename}",
        f"../data/benchmark/{filename}",
        f"llm-graph-ontology/data/benchmark/{filename}",
        f"../llm-graph-ontology/data/benchmark/{filename}",
        f"Fine-Tuning-PoCs/llm-graph-ontology/data/benchmark/{filename}",
        f"/content/Fine-Tuning-PoCs/llm-graph-ontology/data/benchmark/{filename}",
        f"/content/llm-graph-ontology/data/benchmark/{filename}"
    ]
    for p in paths_to_try:
        if os.path.exists(p):
            with open(p, "r", encoding="utf-8") as f:
                return json.load(f)
    # Fallback search
    for s_root in [os.getcwd(), "/content", "/content/Fine-Tuning-PoCs"]:
        if os.path.exists(s_root):
            for root, dirs, files in os.walk(s_root):
                if filename in files:
                    with open(os.path.join(root, filename), "r", encoding="utf-8") as f:
                        return json.load(f)
    raise FileNotFoundError(f"Could not locate '{filename}'.")

legal_task_a_data = load_legal_benchmark_file("legal_task_a_triplets.json")
legal_task_b_data = load_legal_benchmark_file("legal_task_b_multihop.json")
legal_task_c_data = load_legal_benchmark_file("legal_task_c_link_pred.json")

print(f"Loaded Legal Task A (Triplet IE)    : {len(legal_task_a_data)} samples")
print(f"Loaded Legal Task B (Multi-Hop)     : {len(legal_task_b_data)} samples")
print(f"Loaded Legal Task C (Link Pred)     : {len(legal_task_c_data)} samples")

Loaded Legal Task A (Triplet IE)    : 50 samples
Loaded Legal Task B (Multi-Hop)     : 50 samples
Loaded Legal Task C (Link Pred)     : 50 samples


### Step 4: Define Inference Helper Functions

In [ ]:
def extract_legal_triples_zero_shot(text_passage):
    prompt_messages = [
        {
            "role": "system",
            "content": (
                "You are a formal corporate litigation knowledge graph extractor. "
                "Extract factual triples strictly conforming to the legal ontology relations: "
                "[FILES_CLAIM_AGAINST, ADJUDICATED_BY, APPLIES_PRECEDENT, ESTABLISHES_REMEDY, LIABLE_FOR]. "
                "Output ONLY a valid JSON list of objects with keys: 'subject', 'predicate', 'object'."
            )
        },
        {
            "role": "user",
            "content": f"Extract legal triples from this text:\n{text_passage}"
        }
    ]
    formatted = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=150, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def query_legal_multihop_zero_shot(context, question):
    prompt_messages = [
        {
            "role": "system",
            "content": "You are a legal knowledge graph deductive reasoning engine. Answer the question using ONLY the facts provided. State the target entity name directly."
        },
        {
            "role": "user",
            "content": f"{context}\n\nQuestion: {question}"
        }
    ]
    formatted = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=50, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def predict_legal_link_zero_shot(head, tail, prompt_text):
    prompt_messages = [
        {
            "role": "system",
            "content": "Predict the single valid ontology relation connecting Head to Tail. Reply with ONLY the relation name."
        },
        {
            "role": "user",
            "content": prompt_text
        }
    ]
    formatted = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=25, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print("Helper functions ready.")

Helper functions ready.


### Step 5: Run Full 50-Sample Batch Evaluation Across All 3 Legal Tasks

In [ ]:
from tqdm import tqdm
import re

# ---------------------------------------------------------
# 1. Batch Evaluate Legal Task A (Triplet Extraction F1)
# ---------------------------------------------------------
print("Running Full Batch on Legal Task A (50 samples)...")
task_a_precision_list = []
task_a_recall_list = []
task_a_f1_list = []

def parse_triples_json(raw_text):
    cleaned = re.sub(r'```(?:json)?', '', raw_text).strip('` \n')
    try:
        data = json.loads(cleaned)
        if isinstance(data, list):
            return set((item.get('subject', '').strip().lower(), item.get('predicate', '').strip().lower(), item.get('object', '').strip().lower()) for item in data if isinstance(item, dict))
    except Exception:
        pass
    return set()

for item in tqdm(legal_task_a_data, desc="Legal Task A Triplet IE"):
    pred_raw = extract_legal_triples_zero_shot(item['input_text'])
    pred_triples = parse_triples_json(pred_raw)
    true_triples = set((t['subject'].strip().lower(), t['predicate'].strip().lower(), t['object'].strip().lower()) for t in item['target_triples'])
    
    tp = len(pred_triples.intersection(true_triples))
    prec = tp / len(pred_triples) if len(pred_triples) > 0 else 0.0
    rec = tp / len(true_triples) if len(true_triples) > 0 else 0.0
    f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    
    task_a_precision_list.append(prec)
    task_a_recall_list.append(rec)
    task_a_f1_list.append(f1)

# ---------------------------------------------------------
# 2. Batch Evaluate Legal Task B (Multi-Hop Deduction Accuracy)
# ---------------------------------------------------------
print('\nRunning Full Batch on Legal Task B (50 samples)...')
hop2_correct = 0
hop3_correct = 0

for item in tqdm(legal_task_b_data, desc="Legal Task B Multi-Hop"):
    ans_2 = query_legal_multihop_zero_shot(item['subgraph_context'], item['query_2hop']).strip().lower()
    ans_3 = query_legal_multihop_zero_shot(item['subgraph_context'], item['query_3hop']).strip().lower()
    
    target_2 = item['target_2hop'].strip().lower()
    target_3 = item['target_3hop'].strip().lower()
    
    if target_2 in ans_2:
        hop2_correct += 1
    if target_3 in ans_3:
        hop3_correct += 1

# ---------------------------------------------------------
# 3. Batch Evaluate Legal Task C (Link Prediction Accuracy)
# ---------------------------------------------------------
print('\nRunning Full Batch on Legal Task C (50 samples)...')
link_correct = 0

for item in tqdm(legal_task_c_data, desc="Legal Task C Link Prediction"):
    pred_rel = predict_legal_link_zero_shot(item['head'], item['tail'], item['prompt']).strip().lower()
    target_rel = item['target_relation'].strip().lower()
    
    if target_rel in pred_rel:
        link_correct += 1

# ---------------------------------------------------------
# 📊 PRINT AGGREGATE SUMMARY TABLE
# ---------------------------------------------------------
avg_prec = sum(task_a_precision_list) / len(task_a_precision_list)
avg_rec = sum(task_a_recall_list) / len(task_a_recall_list)
avg_f1 = sum(task_a_f1_list) / len(task_a_f1_list)
hop2_acc = hop2_correct / len(legal_task_b_data)
hop3_acc = hop3_correct / len(legal_task_b_data)
link_acc = link_correct / len(legal_task_c_data)

print('\n' + '=' * 65)
print('📊 OFFICIAL LEGAL DOMAIN ZERO-TRAINING BASELINE SUMMARY (50 SAMPLES)')
print('=' * 65)
print(f'Task A: Legal Triplet IE Precision      : {avg_prec * 100:.2f}%')
print(f'Task A: Legal Triplet IE Recall         : {avg_rec * 100:.2f}%')
print(f'Task A: Legal Triplet IE Macro F1       : {avg_f1 * 100:.2f}%')
print('-' * 65)
print(f'Task B: 2-Hop Precedent Accuracy        : {hop2_acc * 100:.2f}%')
print(f'Task B: 3-Hop Remedy Deduction Accuracy : {hop3_acc * 100:.2f}%')
print('-' * 65)
print(f'Task C: Legal Link Prediction Accuracy  : {link_acc * 100:.2f}%')
print('=' * 65)

legal_baseline_results = {
    "domain": "CorporateLitigationOntology",
    "model": "Qwen/Qwen2.5-1.5B-Instruct (Untuned Zero-Shot)",
    "task_a_precision": round(avg_prec, 4),
    "task_a_recall": round(avg_rec, 4),
    "task_a_f1": round(avg_f1, 4),
    "task_b_hop2_accuracy": round(hop2_acc, 4),
    "task_b_hop3_accuracy": round(hop3_acc, 4),
    "task_c_link_accuracy": round(link_acc, 4)
}
with open("legal_zero_training_baseline_results.json", "w", encoding="utf-8") as f:
    json.dump(legal_baseline_results, f, indent=2)
print("Saved legal baseline metrics to legal_zero_training_baseline_results.json")

Running Full Batch on Legal Task A (50 samples)...


Legal Task A Triplet IE: 100%|██████████| 50/50 [04:01<00:00,  4.83s/it]



Running Full Batch on Legal Task B (50 samples)...


Legal Task B Multi-Hop: 100%|██████████| 50/50 [00:48<00:00,  1.04it/s]



Running Full Batch on Legal Task C (50 samples)...


Legal Task C Link Prediction: 100%|██████████| 50/50 [00:23<00:00,  2.15it/s]


📊 OFFICIAL LEGAL DOMAIN ZERO-TRAINING BASELINE SUMMARY (50 SAMPLES)
Task A: Legal Triplet IE Precision      : 30.67%
Task A: Legal Triplet IE Recall         : 21.33%
Task A: Legal Triplet IE Macro F1       : 25.07%
-----------------------------------------------------------------
Task B: 2-Hop Precedent Accuracy        : 92.00%
Task B: 3-Hop Remedy Deduction Accuracy : 86.00%
-----------------------------------------------------------------
Task C: Legal Link Prediction Accuracy  : 24.00%
Saved legal baseline metrics to legal_zero_training_baseline_results.json
